# Analyze object motility

**Purpose.** Generate and track time-series masks, then quantify trajectory, velocity, displacement, and infection-related quality control.

**Recommended use.** Use for time-series acquisitions in which movement is the primary phenotype and tracked merged arrays have not yet been generated.

**Primary outputs.** Tracked masks, per-track measurements, well-level motility summaries, and quality-control figures.

**Desktop route.** Mask → Timelapse, followed by Measure → Motility Assay

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry points

This workflow calls the following public functions in sequence:

- [`spacr.core.preprocess_generate_masks_timelapse`](https://einarolafsson.github.io/spacr/api/spacr/core/index.html#spacr.core.preprocess_generate_masks_timelapse)

```python
preprocess_generate_masks_timelapse(preprocess_generate_masks_timelapse_settings)
```

- [`spacr.timelapse.automated_motility_assay`](https://einarolafsson.github.io/spacr/api/spacr/timelapse/index.html#spacr.timelapse.automated_motility_assay)

```python
automated_motility_assay(automated_motility_assay_settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.core import preprocess_generate_masks_timelapse
from spacr.timelapse import automated_motility_assay

## 3. Settings and API reference

Review the parameter definitions here, then edit the values in the categorized code cells below. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the installed spaCR version so that the notebook remains aligned with the public API.

### [`spacr.core.preprocess_generate_masks_timelapse`](https://einarolafsson.github.io/spacr/api/spacr/core/index.html#spacr.core.preprocess_generate_masks_timelapse)

> Organelle parameters are placed in a separate code cell to keep the primary segmentation configuration readable.


#### Input & Metadata

- **`src`** *(required)* — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`cell_channel`** *(conditionally required)* — (int or None) - Zero-indexed raw acquisition channel that Cellpose segments into cell masks; it also selects which channel the cell_background, cell_Signal_to_noise and remove_background_cell settings are applied to during preprocessing. Set to None and no cell masks, cell table or cell crops are produced. At least one of cell/nucleus/pathogen/organelle_channel must be an integer or the run aborts. Default None.
- **`nucleus_channel`** *(conditionally required)* — (int or None) - Zero-indexed raw acquisition channel segmented into nucleus masks, and the channel that nucleus_background, nucleus_Signal_to_noise and remove_background_nucleus apply to. None means no nucleus masks, hence no nucleus table, no cell-to-nucleus linking, and nothing subtracted from the cytoplasm mask. Set it whenever a DNA stain was acquired. Default None.
- **`pathogen_channel`** *(conditionally required)* — (int or None) - Zero-indexed raw acquisition channel segmented into pathogen masks (Toxoplasma etc.), and the channel pathogen_background, pathogen_Signal_to_noise and remove_background_pathogen apply to. None disables pathogen segmentation, the pathogen table, the infected-only filter (uninfected) and the adjust_cells step, which needs cell, nucleus and pathogen masks together. Default None.
- **`channels`** *(optional)* — (list of int) - Zero-indexed image channels kept in merged/*.npy and measured by measure_crop; each entry produces its own &lt;object&gt;_channel_&lt;n&gt;_* intensity columns. The list length fixes where masks land, so cell/nucleus/pathogen_mask_dim must shift if you change it. Preprocessing silently resets it to range(n) when it does not match the number of channel folders found. Default [0,1,2,3].
- **`magnification`** *(optional)* — (int) - Objective magnification, used only to derive expected object sizes: pixel diameter is 2*mag+80 for cells, 0.75*mag+45 for nuclei and mag for pathogens, with min/max area limits of diameter^2/4 and diameter^2*10. Explicit cell_diameter, nucleus_diameter or pathogen_diameter override it. Set this to the acquisition objective magnification (10, 20, 40 or 60). Default 20.
- **`metadata_type`** *(optional)* — (str) - Filename convention used to parse raw images. 'cellvoyager' (default) and 'cq1' use built-in regular expressions, 'custom' uses custom_regex, and 'auto' first renames the folder to Yokogawa naming (using custom_regex when supplied, otherwise automatic detection) before parsing. An incorrect convention can misassign plate, well, field, or channel identifiers and place images in incorrect channel folders.
- **`custom_regex`** *(optional)* — (str or None) - Python regex with named groups that extracts metadata from raw image filenames. It must supply wellID, fieldID and chanID; plateID is optional (falling back to the source folder name), and timeID or sliceID may be absent. With metadata_type='custom', a filename that does not match or lacks a required group is skipped with a warning, which can reduce the dataset. With 'auto', the regex is tried first for Yokogawa renaming and requires only wellID; automatic detection is used if it fails. Default None.

#### Acquisition & Axes

- **`z_stack`** *(optional)* — (bool) - When True, spaCR requires the array to contain an explicit z dimension and raises an error instead of inferring the axis; this enables z_segmentation_mode, anisotropy and stitch_threshold. Standard ingestion collapses z by maximum-intensity projection while organising raw files, so its output has no z axis to segment; supply volumetric arrays directly to spacr.zstack instead. When False, no z-stack code runs and masks match a two-dimensional run. Default False.
- **`z_segmentation_mode`** *(optional)* — (str) - How the z dimension is handled. The three modes answer different questions and their masks are not comparable, so the choice is recorded alongside them. 'project' collapses the stack with z_projection and segments one plane; it is the only mode the Measure module can consume. 'stitch' segments each plane in 2-D and links labels through the stack. 'volumetric' segments the 3-D volume directly and requires anisotropy or voxel sizes. Default 'project'.
- **`z_axis`** *(optional)* — (int or None) - Axis of the incoming array that holds z, specified as 0, 1 or 2. None infers it from shape only when one axis is clearly shorter than the other two, such as a 21x512x512 or 512x512x21 stack. An ambiguous shape such as 64x64x64 raises an error because an incorrect axis segments a transposed volume and produces invalid masks. Set this explicitly whenever the acquisition shape is ambiguous. Default None.
- **`z_projection`** *(optional)* — (str or None) - Method used to collapse z when z_segmentation_mode is 'project'. 'max' retains the brightest value along the stack and is appropriate for sparse fluorescent objects; 'mean' suppresses noise but dilutes signal present in few planes; 'sum' preserves total signal; and 'best_focus' retains only the sharpest plane, which is preferable when one plane is in focus and a maximum-intensity projection would include substantial out-of-focus signal. Ignored by the other modes. Default 'max'.
- **`anisotropy`** *(optional)* — (float or None) - Ratio of z step to xy pixel size (dz / dxy), used by volumetric mode to represent inter-plane distance. A value of 1.0 on a confocal stack whose z step is 3-10 times the xy pixel size can fuse objects along z. Leave this None and set voxel_size_z_um / voxel_size_xy_um to derive it; if neither is available, volumetric mode raises an error rather than assuming 1.0. Measure also uses it for 3-D region properties and distance transforms. Default None.
- **`voxel_size_z_um`** *(optional)* — (float or None) - Spacing between consecutive z planes in micrometres, obtained from the acquisition metadata. Together with voxel_size_xy_um it determines anisotropy and converts object volumes from voxel counts into cubic micrometres. Changing it rescales every physical z quantity and the anisotropy used for segmentation; it has no effect on a 'project' run. Measure uses the pair to report 3-D morphology in physical units and records the units in measurement_units. Default None.
- **`voxel_size_xy_um`** *(optional)* — (float or None) - Width of one pixel in micrometres in the image plane, assumed square. Used with voxel_size_z_um to derive anisotropy and to turn voxel counts into physical volumes and surface areas. Note this is a different setting from um_per_pixel, which only sizes the scale bar drawn on figures and never reaches a measurement. This one does reach measurements, but only on a 3-D run: a 2-D run never applies it, because doing so would turn every *_area from px2 into um2 under an unchanged column name. Default None.
- **`stitch_threshold`** *(optional)* — (float) - Minimum overlap, as an intersection-over-union between 0 and 1, for a label in one plane to be treated as the same object as a label in the plane below when z_segmentation_mode is 'stitch'. Raising it splits objects that drift or change shape between planes into several shorter ones; lowering it fuses neighbouring objects that merely overlap in projection. Matching is one-to-one, so when two objects both overlap the same object below only the better match inherits its label and the other starts a new one. Ignored by the other two modes. Default 0.25.

#### Image Preprocessing

- **`normalize`** *(optional)* — (bool) - Percentile-normalize each image channel from the 2nd to 98th percentile, clipped to 0-1, before display or model input. In the activation-map tool this rescales the image beneath the CAM or saliency heatmap. Enable it when raw channels are too dim under the overlay. It affects display and input scaling, not stored pixels. Default True.
- **`lower_percentile`** *(optional)* — (float) - Percentile of the non-zero pixels in each channel used as the low anchor when rescaling that channel to 0-1; the high anchor is chosen automatically between the 98th and 99.5th percentile. Raise it to crush more dim background to black, lower it to preserve faint signal. Valid 0-100, default 2.
- **`randomize`** *(optional)* — (bool) - Shuffle the order of the per-field arrays before they are grouped into normalization batches, so each batch spans plates and wells instead of one acquisition block - this matters because normalization percentiles are computed per batch. Forced to False for timelapse runs to keep frames in sequence. Default True.
- **`batch_fields`** *(optional)* — (int) - Streaming pipeline only (pipeline_style='v2'): how many whole field stacks are loaded into RAM before one Cellpose batch is segmented. Larger values keep the GPU busier and cut the number of read passes over the plate, at a memory cost of roughly one full field stack each. Ignored entirely by the v1 pipeline. Default 8.
- **`consolidate`** *(optional)* — (bool) - Before processing, recursively scan src for images and copy them into a single &lt;src&gt;/consolidated folder, prefixing each filename with its subfolder names so nothing collides; src is then repointed there. Use it when one plate's images are split across per-well or per-channel subfolders. Copies, so disk use roughly doubles. Default False.
- **`denoise`** *(optional)* — (bool) - Legacy denoising toggle for the mask pipeline. This key is not read and has no effect. To enable denoising, set the per-object restore settings (cell_restore_type, nucleus_restore_type, or pathogen_restore_type) to 'denoise', which routes segmentation through Cellpose's CellposeDenoiseModel. Default False.

#### Cell Segmentation

- **`cell_model_name`** *(optional)* — (str) - Cell-segmentation weights. Cellpose 4 provides the stock 'cpsam' model; alternatively, provide a CPSAM checkpoint created by Train Cellpose, loaded as pretrained_model. Legacy names ('cyto', 'cyto2', 'cyto3', 'nuclei') remain accepted but resolve to cpsam because Cellpose 4 no longer ships those models. Only diameter changes inference (scaling by 30/diameter); model_type and diam_mean are not used in v4.0.1+. Default 'cpsam'.
- **`cell_diameter`** *(optional)* — (int or None) - Expected cell diameter in pixels. Cellpose 4 rescales the image by 30/diameter before segmentation, aligning the expected object size with the scale used to train CPSAM; leave it None to segment at native scale. Set it when cells are much larger or smaller than ~30 px and segmentation produces fragmented or merged masks. spacr.diameter.estimate_diameters estimates a value from the selected fields. Default None.
- **`cell_CP_prob`** *(optional)* — (float) - Cellpose cellprob_threshold: only pixels whose predicted cell probability exceeds it are assigned to a mask. Raise it to shrink outlines and drop faint or spurious cells; lower it to grow outlines and recover dim ones. Valid range roughly -6 to 6, default 0. Lower it first when whole cells are missing.
- **`cell_FT`** *(optional)* — (float) - Cellpose flow_threshold: the maximum allowed error between a candidate mask's recomputed flows and the network's predicted flows. Masks above it are discarded, so lowering it strips ragged or implausible cells but also loses real ones; raising it keeps more. Usable range about 0-3 (GUI allows -1 to 3). Default 1.0.
- **`adjust_cells`** *(optional)* — (bool) - After segmentation, merge cell labels that divide a single pathogen or nucleus, and absorb an anucleate cell fragment into the neighbouring label with which it shares the largest perimeter. Requires cell, nucleus, and pathogen channels and is skipped for timelapse runs. Enable when large infected cells are systematically fragmented by segmentation. Default False.

#### Nucleus Segmentation

- **`nucleus_model_name`** *(optional)* — (str) - Weights used to segment nuclei. Valid values are 'cpsam' or a path to a custom CPSAM checkpoint produced by Train Cellpose. The legacy values 'nuclei' and 'nucleus' are accepted for compatibility and mapped to 'cpsam' because Cellpose 4 removed the pre-SAM models. Configure nucleus_diameter to control scale; of the three parameters that previously distinguished models, only diameter remains operational (eval rescales by 30/diameter), while model_type and diam_mean are logged as 'not used in v4.0.1+' and omitted. Default 'cpsam'.
- **`nucleus_diameter`** *(optional)* — (int or None) - Expected nucleus diameter in pixels, used by Cellpose 4 to rescale the image by 30/diameter before segmentation. None segments at native scale. Because nuclei are commonly the smallest segmented objects, this parameter often requires explicit configuration for low-magnification acquisitions. spacr.diameter.estimate_diameters estimates a value. Default None.
- **`nucleus_CP_prob`** *(optional)* — (float) - Cellpose cell-probability threshold for the nucleus channel, passed straight to model.eval as cellprob_threshold. A pixel must exceed it to join a mask, so raising it shrinks masks and drops dim nuclei, while lowering it grows masks and recovers faint ones along with more debris. Useful range about -6 to 6; default 0.
- **`nucleus_FT`** *(optional)* — (float) - Cellpose flow_threshold for nucleus masks: the maximum allowed error between a mask's recomputed flows and the network's predicted flows. Lowering it discards more irregularly shaped nuclei, giving fewer but cleaner objects; raising it keeps nearly everything Cellpose proposes. Typical range 0 to 3; spaCR default 1.0, which is permissive.

#### Pathogen Segmentation

- **`pathogen_model_name`** *(optional)* — (str) - Which weights segment pathogens. 'cpsam' or a path to your own Train Cellpose checkpoint. The bundled toxo_pv_lumen / toxo_cyto checkpoints were Cellpose-3 CPnet and cannot load into CPSAM's transformer, so they are mapped to 'cpsam' and reported. The older 'pathogen_model' key still overrides this one when set. Of the three parameters that used to distinguish models only diameter still acts (eval rescales by 30/diameter); model_type and diam_mean are logged 'not used in v4.0.1+' and dropped. Default 'cpsam'.
- **`pathogen_diameter`** *(optional)* — (int or None) - Expected pathogen diameter in pixels, used by Cellpose 4 to rescale the image by 30/diameter before segmenting. None segments at native scale. Intracellular parasites are often only a few pixels across at low magnification, where rescaling matters most. spacr.diameter.estimate_diameters proposes a value. Default None.
- **`pathogen_CP_prob`** *(optional)* — (float) - Cellpose cellprob_threshold for the pathogen channel: a pixel is claimed by a mask only if its predicted object probability exceeds this. Lower it (toward -6) to recover dim or small parasites and grow mask boundaries; raise it (toward 6) to shrink masks and drop faint objects. Useful range about -6 to 6. Default 0.
- **`pathogen_FT`** *(optional)* — (float) - Cellpose flow_threshold for pathogen masks: a candidate mask is discarded when its recomputed flows disagree with the network prediction by more than this. Raise it to keep more, sometimes misshapen, parasites; lower it toward 0.4 (Cellpose's own default) to keep only clean, well-formed objects. Typical range 0.0-3.0. Default 1.0.
- **`pathogen_model`** *(optional)* — (str or None) - Path to a custom Cellpose checkpoint used to detect pathogen objects, overriding pathogen_model_name when set. It must be a CPSAM-architecture checkpoint (one your own Train Cellpose run produced); a Cellpose-3 CPnet file cannot load into Cellpose 4. A path that does not exist stops the run rather than falling back to the stock weights silently. Default None.

#### Image Preprocessing (per object)

- **`cell_background`** *(optional)* — (int) - Background intensity of the cell channel in raw image units. Pixels below it are zeroed when remove_background_cell is True, and it is multiplied by cell_Signal_to_noise to set the intensity the normalisation ceiling must reach. Set it from a genuinely empty region; too high and dim cells are erased. Default 100.
- **`cell_Signal_to_noise`** *(optional)* — (int) - Multiplied by cell_background to define the minimum intensity for the normalisation ceiling. spaCR evaluates the 98th through 99.5th percentiles of the cell channel and uses the first value at or above that product as the upper anchor. Increase it to raise the ceiling and reduce normalised intensity; decrease it to increase the visibility of faint cells. Default 10.
- **`remove_background_cell`** *(optional)* — (bool) - Before normalisation, zero every pixel in the cell channel below cell_background. This flattens haze so the percentile stretch is driven by real signal, but it also erases genuinely dim cell edges and can shrink masks. Enable only once cell_background is set from an actual empty region. Default False.
- **`nucleus_background`** *(optional)* — (int) - Raw intensity value treated as background in the nucleus channel. When remove_background_nucleus is True, every pixel below it is zeroed before normalization; it is also multiplied by nucleus_Signal_to_noise to set the upper-clip target. Raise it for images with high offset or autofluorescence, lower it if dim nuclei disappear. Default 100.
- **`nucleus_Signal_to_noise`** *(optional)* — (float) - Multiplied by nucleus_background to define the intensity a bright pixel must reach before normalisation stops increasing the upper clip point. spaCR evaluates the 98th through 99.5th percentiles of the non-zero nucleus channel and uses the first that reaches the threshold, with the 99.5th percentile as the fallback. A higher value raises the clip point, reduces contrast stretching and protects bright nuclei from saturation; a lower value increases contrast for dim nuclei but saturates bright nuclei sooner. Default 10.
- **`remove_background_nucleus`** *(optional)* — (bool) - Before normalizing the nucleus channel, zero every pixel below nucleus_background and exclude those pixels from the percentile calculation. Enabling it raises contrast on real nuclei and suppresses haze, but clips genuinely dim nuclei to zero so they may become unsegmentable. Default False; check nucleus_background against raw images first.
- **`pathogen_background`** *(optional)* — (int) - Assumed background intensity of the pathogen channel in raw image units. It has two jobs: when remove_background_pathogen is True every pixel below it is zeroed, and it is multiplied by pathogen_Signal_to_noise to set the brightness the normalisation ceiling must reach. Raise it if dim haze is being segmented; lower it if faint parasites vanish. Default 100.
- **`pathogen_Signal_to_noise`** *(optional)* — (int) - Expected foreground-to-background ratio of the pathogen channel. Multiplied by pathogen_background, it defines the minimum intensity for the normalisation ceiling. spaCR evaluates percentiles 98 through 99.5 and uses the first that reaches the threshold, with the 99.5th percentile as the fallback. Increase it for a higher ceiling with less clipping; decrease it for greater contrast. Default 10.
- **`remove_background_pathogen`** *(optional)* — (bool) - Before normalising the pathogen channel, hard-zero every pixel whose raw intensity is below pathogen_background. Enable it when diffuse autofluorescence inflates the low percentile and Cellpose starts segmenting haze; leave it off for dim parasites, since the clipping erases real signal and biases downstream intensity measurements. Default False.

#### Object Filtration (all objects)

- **`cell_min_area`** *(optional)* — (int) - Minimum cell area in pixels^2. Passed to Cellpose as min_size so undersized masks are dropped during segmentation, then re-applied afterwards to delete any object below it. Raise it to clear debris and fragments; set it too high and genuine small cells disappear. 0 disables. Default 0.
- **`cell_max_area`** *(optional)* — (int or None) - Maximum cell area in pixels^2; objects larger than this are deleted after segmentation. Use it to discard clumps or debris blobs that Cellpose labelled as one huge cell. It only deletes, it never splits - use cell_intensity_split for that. 0 or None disables the filter. Default 0.
- **`cell_min_object_area`** *(optional)* — (int) - Absolute pixel-area floor for splitting: the split threshold is the larger of cell_area_multiplier times the median cell area and this value, so cells at or below it are never cut. Raise it to protect small cells in fields where the median area is low. Ignored unless cell_intensity_split is True. Default 100.
- **`cell_area_multiplier`** *(optional)* — (float) - Splitting threshold for cell_intensity_split: only cells whose area exceeds this multiple of the median cell area in the image (or cell_min_object_area, whichever is larger) are watershed-split. Lower it toward 1.5 to include borderline aggregates; raise it to restrict splitting to larger candidate doublets. Ignored unless cell_intensity_split is True. Default 2.0.
- **`cell_min_distance`** *(optional)* — (int) - Minimum separation in pixels between watershed seeds when cell_intensity_split divides an oversized cell; seeds are local maxima of the distance transform. Increase it to produce fewer, larger fragments and reduce over-segmentation of individual cells; decrease it to separate tightly packed cells. Ignored unless cell_intensity_split is True. Default 10.
- **`cell_perimeter_fraction`** *(optional)* — (float) - For each touching pair of cell labels, the shared boundary length divided by the smaller object's perimeter; pairs at or above this fraction are merged into one cell. Low values such as 0.1 merge aggressively and can fuse true neighbours, high values only rejoin pieces of the same cell. 0 disables perimeter merging. Default 0.
- **`cell_remove_border_objects`** *(optional)* — (bool) - Delete every cell label touching any of the four image edges before measurement. Removes partial cells whose area and total intensity are truncated and would bias per-cell statistics, at the cost of losing objects - a large cost in fields where cells are big relative to the field. Default False.
- **`nucleus_min_area`** *(optional)* — (int) - Minimum nucleus area in pixels^2, applied twice: passed to Cellpose as min_size so small masks are never emitted, then re-applied to the label image so any surviving object below it is deleted and the rest renumbered. Raise it to drop debris and fragments. 0 (default) disables both filters.
- **`nucleus_max_area`** *(optional)* — (int or None) - Maximum nucleus area in pixels^2; after segmentation, labels larger than this are deleted and the remaining nuclei are renumbered. Use it to remove unsplit clumps of touching nuclei or large segmentation artifacts covering a substantial fraction of the field. 0 (the default) or None disables the filter.
- **`nucleus_min_object_area`** *(optional)* — (int) - Absolute floor in pixels^2 on the watershed split threshold: objects at or below it are never split, even when nucleus_area_multiplier times the field's median area would fall lower. Raise it to protect small nuclei in fields where the median object is tiny. Default 100; used only when nucleus_intensity_split is True.
- **`nucleus_area_multiplier`** *(optional)* — (float) - Splitting threshold expressed as a multiple of the median nucleus area in each field: only objects larger than this multiple (and larger than nucleus_min_object_area) are candidates for watershed splitting. Lower it toward 1.5 to split more aggressively; raise it to restrict splitting to larger nuclear aggregates. Default 2.0; used only when nucleus_intensity_split is True.
- **`nucleus_min_distance`** *(optional)* — (int) - Minimum separation in pixels between watershed seeds when dividing oversized nucleus labels; seeds are local maxima of the object's distance transform. Set it near the radius of one nucleus: values that are too small over-segment nuclei, while values that are too large yield a single seed and prevent splitting. Default 10; used only when nucleus_intensity_split is True.
- **`nucleus_perimeter_fraction`** *(optional)* — (float) - Merge two touching nucleus labels when their shared boundary covers at least this fraction of the smaller object's perimeter. Low non-zero values merge aggressively (0.1 joins barely-touching nuclei); high values only fuse objects sharing most of an edge. Range 0-1; 0 (default) disables perimeter merging. Use it when one nucleus is split into fragments.
- **`nucleus_remove_border_objects`** *(optional)* — (bool) - After segmentation, delete every nucleus label touching any of the four image edges, then renumber the rest. Enable it when measuring nucleus area or total intensity, since clipped nuclei bias those downward; leave it off for counts or positions, as it discards real objects at every field boundary. Default False.
- **`pathogen_min_area`** *(optional)* — (int) - Minimum pathogen area in pixels squared. Passed to Cellpose as min_size so undersized masks never leave segmentation, then re-applied in the merge/split/filter pass. 0, the default, disables it. Raise it to clear speckle and debris; set it too high and small or newly divided parasites disappear.
- **`pathogen_max_area`** *(optional)* — (int or None) - Maximum pathogen area in pixels squared; labels larger than this are deleted after segmentation. 0, the default, or None disables the filter. Use it to remove fused clumps and large segmentation artifacts that would otherwise dominate per-object statistics; use pathogen_intensity_split instead when clumps should be separated rather than discarded.
- **`pathogen_min_object_area`** *(optional)* — (int) - Absolute floor in pixels squared below which a pathogen label is never split, whatever the median area says: the effective split threshold is max(pathogen_area_multiplier x median area, this value). Raise it to protect small parasites from fragmentation in sparse fields. Used only when pathogen_intensity_split is True. Default 100.
- **`pathogen_area_multiplier`** *(optional)* — (float) - Splitting threshold expressed as a multiple of the median pathogen area in the field. Only labels above this threshold and pathogen_min_object_area are processed by watershed segmentation. Lower values, such as 1.5, split more objects; higher values restrict splitting to larger aggregates. Used only when pathogen_intensity_split is True. Default 2.0.
- **`pathogen_min_distance`** *(optional)* — (int) - Minimum separation in pixels between watershed seeds when splitting oversized pathogen labels; seeds are local maxima of the distance transform. Raise it for fewer, larger fragments (or none, leaving the object intact); lower it to cut clumps into more pieces. Used only when pathogen_intensity_split is True. Default 10.
- **`pathogen_perimeter_fraction`** *(optional)* — (float) - Fraction, from 0 to 1, of the smaller label's perimeter that two touching pathogen objects must share before they are merged. The default of 0 disables perimeter-based merging. Values near 0.1 merge most touching objects, whereas values from 0.5 to 0.8 merge only objects with a long shared boundary. Use this setting to join vacuoles that Cellpose divided into multiple labels.
- **`pathogen_remove_border_objects`** *(optional)* — (bool) - Delete any pathogen label touching the first or last row or column of the image. Enable it so partially imaged parasites do not enter area and intensity statistics with truncated values; leave it off when parasites are sparse and losing edge objects costs too much data. Default False.

#### Intensity Handling (all objects)

- **`cell_intensity_merge`** *(optional)* — (bool) - Merge touching cell labels when the mean intensity along their shared boundary is at least as high as the interior intensity of the dimmer label, indicating no detectable membrane boundary between them. This can correct over-segmentation of individual cells. The comparison statistic is set by cell_intensity_threshold_method. Default False.
- **`cell_intensity_split`** *(optional)* — (bool) - Split oversized cell labels by distance-transform watershed before the merge and filter steps. Objects larger than cell_area_multiplier times the median cell area are seeded at local distance maxima cell_min_distance apart and cut. Despite the name no intensity is used. Enable when several touching cells share one label. Default False.
- **`cell_intensity_percentile`** *(optional)* — (int) - Percentile from 0 to 100 of the dimmer cell's interior intensity used as the merge reference when cell_intensity_threshold_method is 'percentile'. Raising it toward 95 sets a higher bar for the shared boundary to clear, so fewer pairs merge; lowering it merges more. Ignored when the method is 'mean'. Default 75.
- **`cell_intensity_threshold_method`** *(optional)* — (str) - Reference statistic that cell_intensity_merge compares the shared-boundary intensity against: 'mean' uses the mean interior intensity of the dimmer of the two cells, 'percentile' uses its cell_intensity_percentile instead. Any value other than 'mean' is treated as 'percentile'. Choose 'percentile' with a high percentile to make merging rarer. Default 'mean'.
- **`cell_min_intensity_percentile`** *(optional)* — (int) - Drops the dimmest cells per field: the mean intensities of all surviving cells are pooled and objects below this percentile (0-100) of that per-image distribution are removed. Being relative, it always removes roughly this share of objects, however bright the field. Use it to shed out-of-focus cells. 0 disables. Default 0.
- **`cell_max_intensity_percentile`** *(optional)* — (int or None) - Drops the brightest cells per field: objects whose mean intensity is above this percentile (0-100) of the per-image distribution of cell mean intensities are removed. Lower it to about 99 to strip saturated blobs and fluorescent debris. Relative, not an absolute intensity. Use 100 to disable. Default 100.
- **`nucleus_intensity_merge`** *(optional)* — (bool) - Merge touching nucleus labels when the mean intensity along their shared boundary is at least as high as the dimmer object's own intensity statistic - i.e. there is no dark seam between them, so the split is spurious. Controlled by nucleus_intensity_threshold_method and nucleus_intensity_percentile. Default False; enable when Cellpose over-segments single nuclei.
- **`nucleus_intensity_split`** *(optional)* — (bool) - Enable watershed splitting of over-large nucleus labels: objects bigger than nucleus_area_multiplier times the field's median nucleus area are cut at distance-transform maxima spaced nucleus_min_distance apart. Despite the name it uses shape and area, not intensity. Default False; enable when clumps of touching nuclei are labelled as one object.
- **`nucleus_intensity_percentile`** *(optional)* — (int) - Percentile of each nucleus's own pixel intensities used as the merge reference when nucleus_intensity_threshold_method is 'percentile'. Higher values (90) demand a very bright shared boundary and merge almost nothing; lower values (50) merge readily. Range 0-100, default 75. Ignored when the method is 'mean'.
- **`nucleus_intensity_threshold_method`** *(optional)* — (str) - Which statistic of the dimmer of two touching nuclei the shared-boundary intensity is compared against when nucleus_intensity_merge is on. 'mean' (default) uses that object's mean intensity; 'percentile' uses its nucleus_intensity_percentile-th percentile, which at the default 75 is stricter and merges fewer pairs. Ignored when nucleus_intensity_merge is False.
- **`nucleus_min_intensity_percentile`** *(optional)* — (int) - Drops the dimmest nuclei per field: spaCR takes the mean nucleus-channel intensity of every object surviving the area and border filters and removes those below this percentile of that per-field distribution. Because it is relative, any value above 0 always removes some objects. Range 0-100; 0 (default) disables it.
- **`nucleus_max_intensity_percentile`** *(optional)* — (int) - Drops the brightest nuclei per field: objects whose mean nucleus-channel intensity exceeds this percentile of the per-field distribution of object means are removed. Useful against saturated debris and staining artefacts. Range 0-100; 100 (the default) disables it, and any lower value always removes some objects.
- **`pathogen_intensity_merge`** *(optional)* — (bool) - Merge two touching pathogen labels when the mean intensity along their shared border is at least as high as the interior intensity of the dimmer label, indicating no detectable intensity minimum between them. This can correct over-segmentation of a single vacuole. Requires an intensity image and is controlled by pathogen_intensity_threshold_method. Default False.
- **`pathogen_intensity_split`** *(optional)* — (bool) - Enable watershed splitting of oversized pathogen labels. Despite the name, the split is geometric: objects larger than max(pathogen_area_multiplier x median area, pathogen_min_object_area) are divided at local maxima of their distance transform. Enable it when several parasites in one vacuole are fused into a single mask. Default False.
- **`pathogen_intensity_percentile`** *(optional)* — (int) - Percentile, 0-100, of a pathogen's interior intensity used as the merge reference when pathogen_intensity_threshold_method is 'percentile'. Two touching labels merge only if their shared border is at least this bright inside the dimmer object, so raising it demands a brighter border and merges fewer pairs. Ignored when the method is 'mean'. Default 75.
- **`pathogen_intensity_threshold_method`** *(optional)* — (str) - How the reference brightness is computed when pathogen_intensity_merge decides whether two touching labels have a real edge. 'mean' compares the shared-border intensity to the mean interior intensity of the dimmer object; 'percentile' compares it to pathogen_intensity_percentile of that object instead, which is stricter and merges fewer pairs. Default 'mean'.
- **`pathogen_min_intensity_percentile`** *(optional)* — (int) - Relative brightness cutoff, 0-100: within each field the mean intensity of every surviving pathogen is ranked, and objects below this percentile of that distribution are deleted. It is not an absolute intensity, so how many objects go depends on the object count. 0, the default, disables it. Raise it to drop dim false positives.
- **`pathogen_max_intensity_percentile`** *(optional)* — (int or None) - Upper end of the same per-field percentile filter: pathogens whose mean intensity exceeds this percentile of the field's pathogen mean intensities are deleted. 100, the default, disables it. Lower it to strip saturated debris and bright artefacts. Any value below 100 forces the intensity image to be loaded during filtering.

#### Quality Control

- **`seg_qc`** *(optional)* — (str) - Segmentation quality control performed when masks are written, before measurement. 'off' skips scoring; 'report' scores every field, writes qc/segmentation_qc_&lt;object&gt;.csv, and displays detected quality issues; 'flag' also writes per-field JSON for downstream processing; 'stop' raises when the plate verdict is 'fail', after writing the scorecard. No mode deletes or omits a field, and 'stop' does not raise for a 'warn' verdict. Default 'report'.
- **`seg_qc_min_objects`** *(optional)* — (int) - Fields with fewer objects than this are classified as near-empty, and robust per-field size statistics are suppressed because the median absolute deviation is unstable for very small samples. Increase the value for confluent cell plates expected to contain hundreds of objects per field; reduce it to 3-5 for low-multiplicity pathogen assays in which few objects per field are expected. Default 10.
- **`seg_qc_count_ratio`** *(optional)* — (float) - Permitted ratio between a field's object count and the plate median before the field is flagged. A value of 0.25 flags counts below one quarter of the median or above its reciprocal, four times the median. Calibrate this threshold with representative control plates when expected object density varies by assay. Default 0.25.
- **`seg_qc_size_ratio`** *(optional)* — (float) - Fold change in a field's median object diameter, measured against the plate median, that marks it as fused or fragmented when its object count has moved in the opposite direction. Merging two equal objects into one increases equivalent diameter by a factor of approximately 1.41, while dividing one object into two produces the reciprocal change; the default therefore reflects the expected geometric ratio. Default 1.4.
- **`seg_qc_border_fraction`** *(optional)* — (float) - Fraction of a field's objects allowed to touch the image edge before the field is flagged. Edge objects are truncated, so their crop dimensions and measured areas are biased downward. Geometry alone places approximately two object diameters on the border, corresponding to about 8% for 60 px cells in a 1400 px field; the default is therefore above the fraction expected in a valid field. Default 0.3.
- **`seg_qc_outlier_mad`** *(optional)* — (float) - Number of robust standard deviations, each defined as 1.4826 times the median absolute deviation, that an object's diameter may differ from the field median before it is classified as a size outlier. Median and MAD limit the influence of debris. The default of five accommodates heavier-tailed biological size distributions; a threshold of three can flag valid objects. Default 5.
- **`seg_qc_outlier_fraction`** *(optional)* — (float) - Fraction of a field's objects that must fall outside the robust size range before the field is reported as containing multiple size populations. Such objects commonly represent debris, fused pairs, or fragments. Decrease the value to increase sensitivity to mixed fields, at the cost of more flags. Default 0.15.
- **`seg_qc_foreground_fraction`** *(optional)* — (float) - Foreground coverage at or above which a field is classified as confluent. The distance-transform fusion check runs only above this threshold. Increasing it reduces computation but decreases sensitivity to fusion in moderately dense fields; decreasing it evaluates more sparse fields and increases runtime. It matches the fused_fraction used by the diameter estimator. Default 0.35.
- **`seg_qc_split_ratio`** *(optional)* — (float) - Minimum ratio of distance-transform maxima to mask objects required to flag fusion in a field already classified as confluent. A value of 2 requires at least two resolved maxima per mask object on average. Increasing the value reduces sensitivity to fused masks. Default 2.
- **`seg_qc_min_diameter`** *(optional)* — (float) - Equivalent diameter in pixels below which an object is treated as a fragment; it drives the over-segmentation check and sets the seed floor of the fusion cross-check. Lower it to two or three for punctate organelles, where five-pixel objects may represent valid signal rather than debris, and raise it for large cells where components of that size are likely segmentation fragments. Default 5.
- **`seg_qc_tiny_fraction`** *(optional)* — (float) - Fraction of a field's objects that may be smaller than seg_qc_min_diameter before the field is classified as over-segmented. Dividing one cell into multiple fragments increases this fraction, whereas a valid field containing limited debris should remain below the default threshold. Default 0.3.
- **`seg_qc_max_object_fraction`** *(optional)* — (float) - Fraction of the field that a single label may cover before it is classified as evidence of fusion rather than a valid object. A component covering one quarter of a field commonly represents a confluent monolayer merged into one mask; the diameter estimator excludes such components for the same reason. Lower it for small objects on large fields; raise it only when a single large object per field is expected. Default 0.25.
- **`seg_qc_plate_fail_fraction`** *(optional)* — (float) - Fraction of failing fields at which the plate-level scorecard changes from warn to fail. The default 0.1 corresponds approximately to one column of a 96-well plate. This setting changes only the reported verdict and does not determine which fields are processed. Default 0.1.

#### Visualization & Diagnostics

- **`plot`** *(optional)* — (bool) - Render and save quality-control figures during the pipeline, including channel montages, Cellpose mask overlays, filtration comparisons, and crop grids. Figure generation increases runtime and memory use, particularly for complete plates. test_mode enables this setting automatically. Default False.
- **`cmap`** *(optional)* — (str) - Matplotlib colormap applied to single-channel image previews and plate heatmaps. Perceptually uniform maps ('viridis', 'inferno', 'magma') preserve the relative visibility of intensity differences; 'gray' resembles the raw single-channel microscope image. Any registered matplotlib name is accepted, with an '_r' suffix to reverse it. Default 'inferno' for image plots and 'viridis' for plate heatmaps.
- **`figuresize`** *(optional)* — (int) - Base figure size in inches; figures are built square as figuresize x figuresize and font sizes are derived from it (legend, axis labels and ticks at 0.75x, overlay text at 0.5x). Raise it when text is unreadable at publication scale, lower it to fit panels on screen. Default 10; cluster grids cap total width at 200 inches.
- **`normalize_plots`** *(optional)* — (bool) - Scale each displayed image to its own intensity range before rendering. This affects displayed figures but not measurements. Disable it when comparing brightness across images because independent scaling removes absolute intensity differences. Default True.
- **`examples_to_plot`** *(optional)* — (int) - How many randomly chosen merged image stacks are rendered as segmentation-overlay previews after mask generation (in timelapse mode, per-channel panels instead). Raise it to check outlines and normalization across more fields of view, at the cost of render time and larger PDFs; 0 skips previews entirely. Default 1.

#### Output & Storage

- **`save`** *(optional)* — (bool or list of bool) - Whether to save masks to disk. Can be a list of three booleans for [cell, nucleus, pathogen] independently. Default varies by module -- False for most, True for the sequencing and regression paths.
- **`delete_intermediate`** *(optional)* — (bool) - Legacy force-cleanup switch. True overrides keep_intermediate and keep_original_images, removing stack/, masks/, the numeric per-channel folders, and the orig/ raw backup after merged/ is built. Cleanup is already the default; enable this setting only when cleanup must override those retention settings. Deletion is skipped unless every field of view reached merged/. Default False.
- **`keep_intermediate`** *(optional)* — (bool) - Keep the intermediate stack/ and masks/ folders after the merged/ arrays are built. Off by default: only merged/ is kept (masks are embedded in merged and recorded in the database).
- **`keep_original_images`** *(optional)* — (bool) - Keep the original raw input images (in orig/). Off by default to save disk space; the pixel data lives in merged/.
- **`save_original_images`** *(optional)* — (bool) - After each batch is MIP-projected and merged into stack/, either move the raw input images into src/orig/ (True) or delete them so the pixels live only in stack/ (False). Set False on large screens where the duplicate raw copy will not fit on disk; the deletion is not reversible. Default True.
- **`keep_npz`** *(optional)* — (bool) - Streaming pipeline only (pipeline_style='v2'): write each in-memory NPZ batch under merged/_scratch/ instead of discarding it, so intermediate data from a failed run can be inspected. This increases disk usage; enable it only for diagnosis. Default False.
- **`filter`** *(optional)* — (bool) - Legacy switch for the old post-Cellpose cleanup pass, which re-ran size/intensity/border filtering and logged '_after_filtration' object counts to the database. The current Cellpose-SAM segmentation path never reads it, so toggling it changes nothing; use the per-object &lt;object&gt;_min_area, &lt;object&gt;_max_area and &lt;object&gt;_perimeter_fraction settings instead. Default False.
- **`merge_pathogens`** *(optional)* — (bool) - Legacy option that merged two touching pathogen labels into one when their shared boundary exceeded 66% of the smaller object's perimeter, so a single PV split by Cellpose counted once. The current Cellpose-SAM path ignores it - use pathogen_perimeter_fraction instead. Default False.

#### Runtime & Reliability

- **`preprocess`** *(optional)* — (bool) - Run image preparation before segmentation: group raw files into per-field channel stacks, optionally subtract background, and percentile-normalize each channel into floating-point arrays. Keep True for unprocessed input; set False only when the normalized arrays already exist, because segmentation requires those arrays. Default True.
- **`masks`** *(optional)* — (bool) - Run Cellpose segmentation for every configured object channel (cell, nucleus, pathogen, organelle) and write label stacks to masks/&lt;object&gt;_mask_stack. False performs preprocessing only, producing normalized arrays without label masks; downstream measurement therefore requires a subsequent segmentation step. Default True.
- **`test_mode`** *(optional)* — (bool) - Run the pipeline on a small random subset instead of the whole folder. Mask generation copies test_images (default 10) complete image sets into &lt;src&gt;/test and works there; measure_crop copies test_nr (default 10) merged arrays into test/merged. Both also force verbose and plot on. Use it to check channel assignment, diameters and thresholds before committing to a full plate. Default False.
- **`test_images`** *(optional)* — (int) - How many plate/well/field image sets are copied into a test/ folder when test_mode is on; every channel file belonging to a chosen set is copied together. Raise it for a broader smoke test, lower it for a faster one. Forced to 1 for timelapse runs so a full sequence stays intact. Default 10.
- **`resume`** *(optional)* — (bool) - Continue an interrupted run from its last validated boundary. Mask revalidates existing mask and merged arrays; Measure accepts only fields complete in every owned table and clears partial rows before retrying; and Format Converter reopens each checkpointed TIFF. These validations reduce the risk of reusing partial output and require additional reads during resumption. Default False.
- **`strict_errors`** *(optional)* — (bool or None) - Error-handling policy for recoverable steps. Off records failures in the run ledger and final summary while continuing with successful items. On raises immediately for setup or configuration errors such as unreadable paths, missing columns or inaccessible databases, preventing partial batch results from invalid inputs. Per-item failures such as one corrupt image remain recoverable under either policy. None defers to $SPACR_STRICT_ERRORS. Default None.
- **`max_failure_rate`** *(optional)* — (float or None) - Fraction of failed items above which the run aborts. For example, 0.2 aborts after more than 20% of items fail. The failure ledger is written to the artifact before the abort. None disables rate-based abortion; failures remain counted and reported, and incomplete artifacts are marked partial. Default None.
- **`dry_run`** *(optional)* — (bool) - Validate settings against the selected data, report the planned operations, and stop before any compute begins. Checks that src contains the expected files, channel and mask-plane indices are valid, and required models, barcode CSV files, or measurements.db files are present. Each problem is reported with a suggested correction, followed by a summary of the planned segmentation, measurement, and output locations. Nothing is written and no model is loaded. Default False.
- **`verbose`** *(optional)* — (bool) - Print the resolved settings table, channel and model choices per object type, row counts per table, and object counts after each filter. It only adds console output; enable it to identify which stage produced an unexpected object count. The default is True for mask, UMAP, screen analysis, barcode mapping and Cellpose training, and False for measure, plotting helpers and regression.
- **`n_jobs`** *(optional)* — (int) - CPU workers for parallel stages: measurement, mask adjustment, DataLoader loading, and the sklearn/UMAP calls where -1 means every core. Raise it to shorten CPU-bound steps until RAM or disk I/O saturates. Note the measure-and-crop pipeline overrides your value with cpu_count()-4. Defaults vary by pipeline: cpu_count()-4, -1, or None.
- **`batch_size`** *(optional)* — (int) - How many images are held and processed together in one pass: field stacks during normalization and Cellpose segmentation, crops per step during classifier training and activation maps. Raising it speeds runs up but increases RAM/VRAM roughly linearly; lower it on out-of-memory errors. Defaults: 50 for mask generation, 64 for training.
- **`pipeline_style`** *(optional)* — (str) - Which mask pipeline runs. 'v1' is the disk-based chain (rename, per-channel folders, npy, npz, mask npy, merged/) that measure, annotate and every downstream tool expect, and is the fully tested path. 'v2' streams from the originals and writes one npy per field with masks appended in place, using roughly 60-80% less disk but producing no .npz. Default 'v1'.
- **`diameter_estimate_n_fields`** *(optional)* — (int) - How many fields spacr.diameter.estimate_diameters reads before it proposes cell_diameter, nucleus_diameter and pathogen_diameter from blob statistics instead of requiring manual estimation. Fields are taken on an even stride across the sorted plate, so rows and columns are both represented rather than the first few wells; each field costs about a second of CPU and loads neither torch nor Cellpose. Increase it to 10–20 when wells are heterogeneous or confidence is low; decrease it to 2–3 for a faster preliminary estimate. Default 5.

#### Input & Metadata

- **`number_of_organelles`** *(optional)* — (int) - How many organelle slots this run has, from 0 to 26. Each slot is an independent object with its own channel, its own type preset and its own copy of every detection setting, named organelle_*, organelleb_*, organellec_* and so on; raising the number generates another slot's settings and lowering it hides the slots above the new number without deleting them. A hidden slot keeps its values, is still written to the settings file, and comes back exactly as it was when the number is raised again, so a smaller number can be tried without losing work. Default 4.
- **`organelle_channel`** *(conditionally required)* — (int) - Zero-indexed raw acquisition channel segmented into organelle masks by whichever organelle_method is chosen (otsu, adaptive, log, dog, ridge, hysteresis, cellpose, unet). Setting it to an integer adds an organelle mask plane to merged/ and unlocks the Organelle setting categories in the GUI; None skips organelle segmentation entirely. Default None.

#### Organelle Segmentation

- **`organelle_type`** *(optional)* — (str) - Organelle morphology used to populate recommended detection settings; explicitly configured values are not overwritten. Options are 'punctate', 'vesicular', 'spherical', 'filamentous', 'tubular', 'reticular', 'cisternal', 'toroidal' and 'crescent'. Morphology alone does not determine the detector: 'vesicular' and 'spherical' also use organelle_diameter because a 200 nm vesicle appears punctate whereas a 2 µm vacuole appears annular. Default 'custom', which applies no recommendations.
- **`organelle_diameter`** *(optional)* — (float) - Deprecated. Expected organelle diameter in pixels. The Cellpose-SAM path used for organelles calls model.eval with diameter=None, and no classical method sizes its kernels from it, so changing this value has no effect on organelle masks. Bound object size with organelle_min_size / organelle_max_size instead. Default 30.
- **`organelle_mask_within_cells`** *(optional)* — (bool) - Zero every pixel outside the cell mask before segmenting, so organelles can only be found inside cells and extracellular debris cannot generate objects. Needs cell_mask_stack/ to already exist alongside the organelle source; if it is missing spacr prints a warning and carries on unmasked rather than failing. Default False.

#### Organelle Segmentation (advanced)

- **`organelle_morphology`** *(optional)* — (str) - Shape family of the target organelle; selects the segmentation pipeline and restricts valid organelle_method values. 'spots' denotes punctate structures such as vesicles and lipid droplets; 'network' denotes filamentous structures such as mitochondria and endoplasmic-reticulum tubules; 'irregular' denotes solid, irregular structures such as Golgi and lysosomes; and 'ring' denotes hollow structures such as endosomes and autophagosomes. An unsupported morphology-method pair raises before image loading. Default 'spots'.
- **`organelle_method`** *(optional)* — (str) - Segmentation backend, validated against organelle_morphology: 'otsu' (one global threshold), 'adaptive' (local threshold), 'log'/'dog' (blob detection), 'ridge' (tubeness filter, network only), 'hysteresis' (dual threshold, network only), 'cellpose' (pretrained model), 'unet' (your own model, network only). Classical methods run on CPU across n_jobs workers; cellpose and unet run on the GPU. Default 'otsu'.
- **`organelle_adaptive_block_size`** *(optional)* — (int) - Side length in pixels of the local neighbourhood used to compute the adaptive threshold; must be odd. Small blocks track fine illumination changes but can carve holes out of large organelles; large blocks behave more like a global threshold. A few times the object diameter is a sensible starting point. Default 51.
- **`organelle_adaptive_offset`** *(optional)* — (float) - Subtracted from each local mean to form the adaptive threshold, so a pixel is foreground when it exceeds local_mean minus this value. Increasing the offset lowers the threshold and produces more foreground; use a small or negative value for stricter segmentation. The value uses raw image-intensity units, so an offset tuned for 16-bit data can oversegment ridge and ring modes, which threshold a 0-1 response. Default 5.
- **`organelle_tophat_radius`** *(optional)* — (int) - Radius in pixels of the disk used for white top-hat filtering before Otsu or adaptive spot thresholding; it removes structures broader than the disk, reducing haze and background. Set it slightly above the largest expected spot: smaller values suppress spots, whereas larger values retain more background. Default 5. Ignored by the LoG and DoG methods.
- **`organelle_watershed_spots`** *(optional)* — (bool) - Split touching spots instead of labelling each connected blob once. Under otsu/adaptive it runs a distance-transform watershed with seeds at least 5 px apart; under log/dog it grows a watershed from each blob centre instead of stamping a disk whose radius comes from that blob's own sigma (round(sigma*sqrt(2)), minimum 1 px). Turn it off when single spots are being fragmented. Default True.
- **`organelle_log_min_sigma`** *(optional)* — (float) - Smallest Gaussian scale searched by LoG blob detection, in pixels; the detected blob radius is about sigma times sqrt(2), so sigma 1 finds roughly 1.4 px radius puncta. Raise it to ignore single-pixel noise, lower it to catch the smallest spots. Default 1; must stay below organelle_log_max_sigma.
- **`organelle_log_max_sigma`** *(optional)* — (float) - Largest Gaussian scale searched by LoG blob detection, in pixels; blob radius is about sigma times sqrt(2), so 10 caps detection near a 14 px radius. Raise it to catch large puncta, at a runtime cost since the filter is evaluated once per scale. Default 10; must exceed organelle_log_min_sigma.
- **`organelle_log_num_sigma`** *(optional)* — (int) - How many Gaussian scales are evaluated between organelle_log_min_sigma and organelle_log_max_sigma. More scales resolve a wider spread of spot sizes, but the filter runs once per scale so runtime grows linearly. Default 10; drop to 3-5 when spot size is uniform and you need speed.
- **`organelle_log_threshold`** *(optional)* — (float) - Minimum LoG/DoG response a local maximum must reach to count as a blob, measured after the image is percentile-normalised to 0-1, so it behaves like a contrast fraction. Decrease it to detect fainter puncta at the cost of additional noise; increase it to retain only brighter puncta. Default 0.01. The 'dog' method also reads this key.
- **`organelle_dog_sigma_low`** *(optional)* — (float) - Smallest Gaussian scale searched by Difference-of-Gaussians blob detection, in pixels; it sets the lower bound on detectable spot size (radius about sigma times sqrt(2)). Raise it to suppress fine noise, lower it to catch the smallest puncta. Default 1.0. The detection cutoff itself comes from organelle_log_threshold, not from a dog-specific key.
- **`organelle_dog_sigma_high`** *(optional)* — (float) - Largest Gaussian scale searched by Difference-of-Gaussians blob detection, in pixels. Scales are stepped up from the low sigma by a factor of 1.6 until this bound, so widening the gap costs more passes but covers a wider range of spot sizes. Raise it to catch larger spots. Default 3.0; must exceed organelle_dog_sigma_low.
- **`organelle_ridge_filter`** *(optional)* — (str) - Which vesselness filter enhances filaments before thresholding: 'frangi' (classic, crisp on well-separated tubules), 'sato' (more tolerant of varying thickness), 'meijering' (tuned for thin neurite-like fibres). All run with black_ridges=False, i.e. bright filaments on a dark background. Default 'frangi'; try 'sato' when frangi drops faint filaments.
- **`organelle_ridge_sigmas`** *(optional)* — (list of float) - Scales in pixels at which the vesselness filter detects tubular structures; each value should approximate the half-width of a filament, and responses are combined across scales. Add larger values to detect thick bundles and retain smaller values for fine tubules. Default [1, 2, 3]; runtime increases approximately in proportion to list length.
- **`organelle_skeletonize`** *(optional)* — (bool) - Reduce each thresholded network to a one-pixel-wide skeleton (dilated by 1 px so it stays connected) and label that instead of the filled filaments. Measured areas then track network length rather than filament thickness. Enable for topology and length analysis, disable to measure filament mass. Default False.
- **`organelle_network_threshold`** *(optional)* — (str) - How the ridge-filter response is binarised: 'otsu' takes one global cut-off from the response histogram, 'adaptive' uses a local threshold (organelle_adaptive_block_size / _offset) and keeps faint filaments in dim regions at the cost of extra background. Only read by organelle_method='ridge'; anything unrecognised falls back to otsu without warning. Default 'otsu'.
- **`organelle_hysteresis_low`** *(optional)* — (float) - Weak threshold for hysteresis segmentation: pixels above it are kept only where they connect to a seed above organelle_hysteresis_high. Values below 1.0 are read as a fraction and converted to that percentile of the smoothed image (0.2 = 20th percentile); 1.0 or above is an absolute intensity. Lower it to trace filaments further into their dim tails. Default 0.2.
- **`organelle_hysteresis_high`** *(optional)* — (float) - Strong threshold that seeds hysteresis segmentation - only components containing a pixel above it survive at all, then they grow outward down to organelle_hysteresis_low. Values below 1.0 are read as a percentile of the smoothed image (0.6 = 60th percentile); 1.0 or above is absolute. Raise it to keep only confidently bright filaments. Default 0.6.
- **`organelle_ring_sigma_inner`** *(optional)* — (float) - Low sigma of the Difference-of-Gaussians band-pass that highlights ring walls, in pixels; set it near the wall thickness so the wall survives the high-pass. Too small and pixel noise is retained, too large and the wall blurs into the lumen and the ring stops being detected as hollow. Default 1.0; must be below organelle_ring_sigma_outer.
- **`organelle_ring_sigma_outer`** *(optional)* — (float) - High sigma of the ring Difference-of-Gaussians band-pass, in pixels; it sets the coarse scale that gets subtracted, so keep it around the ring's outer radius. Widen the gap from organelle_ring_sigma_inner to enhance larger rings, narrow it for tight vesicles. Default 3.0; must exceed organelle_ring_sigma_inner.
- **`organelle_ring_min_prominence`** *(optional)* — (float) - Shape gate for ring mode: for each filled object spacr computes abs(mean wall intensity minus mean lumen intensity) divided by the object's mean intensity, and deletes anything below this value. Raise it to keep only clearly hollow objects, lower it to also accept partly filled ones. 0 disables the gate. Default 0.1.
- **`organelle_ring_fill_method`** *(optional)* — (str) - How detected ring walls become solid objects: 'flood' fills every background component that does not touch the image border - accurate, but leaks through any gap in the wall - while 'convex' takes the convex hull of each wall component, which tolerates broken rings but overshoots concave shapes. Default 'flood'; switch to 'convex' when rings come out unfilled.
- **`organelle_morph_radius`** *(optional)* — (int) - Radius in pixels of the disk used for morphological cleanup. In irregular mode it also sets the pre-smoothing sigma (radius/2) and drives a closing then an opening, bridging gaps and erasing protrusions thinner than the disk; network modes use half this radius for closing only. Raise it to smooth ragged outlines, lower it to preserve fine detail. Default 3.
- **`organelle_fill_holes`** *(optional)* — (int) - Fill interior holes up to this area in square pixels after thresholding, preventing a darker centre from creating a ring-shaped segmentation artifact. This is applied only in irregular mode. Increase it when large organelles are incorrectly hollow; use a low value or 0 when a hollow centre is biologically expected. Default 64.
- **`organelle_model_name`** *(optional)* — (str) - Cellpose model used when organelle_method='cellpose'. Cellpose 4 provides only 'cpsam'; the pre-SAM names are accepted and mapped to it. Change this only to point at a custom CPSAM-architecture checkpoint. Of the three parameters that used to distinguish models only diameter still acts (eval rescales by 30/diameter); model_type and diam_mean are logged 'not used in v4.0.1+' and dropped. Default 'cpsam'.
- **`organelle_CP_prob`** *(optional)* — (float) - Cellpose cellprob_threshold. Pixels whose predicted probability of belonging to an object fall below it are excluded, so raising it shrinks masks and drops faint organelles, while lowering it grows masks and recovers dim ones along with more false positives. Useful range roughly -6 to 6. Default 0.0.
- **`organelle_FT`** *(optional)* — (float) - Cellpose flow_threshold: maximum error allowed between a candidate mask's flows and the network prediction. Lower values discard more irregular masks; higher values retain more irregular objects. Increase it when valid non-round organelles are being discarded. Default 0.4.
- **`organelle_resample`** *(optional)* — (bool) - Deprecated. Passed to Cellpose as resample: when True the flows are recomputed at full resolution instead of on the downsampled grid, giving smoother and slightly more accurate outlines for a little extra time. Still forwarded to model.eval on the organelle path. Default True; retain the default unless reduced runtime is required.
- **`organelle_unet_model_path`** *(optional)* — (str or None) - Path to a serialised PyTorch model used when organelle_method='unet'. It must be a torch.load-able whole module, not a state_dict, and take z-scored (B,1,H,W) input returning (B,1,H,W) logits; extra output channels are silently ignored except the first. A missing or invalid path raises before segmentation starts. Default None.
- **`organelle_unet_threshold`** *(optional)* — (float) - Probability cut-off applied to the U-Net's sigmoid output, range 0-1. Lower it to grow the predicted network and recover faint branches at the cost of false positives; raise it to keep only confident pixels, which tends to break weak connections. Objects below organelle_min_size are still removed afterwards. Default 0.5.
- **`summarize_organelles_by`** *(optional)* — (str, list or None) - Parent compartments to roll every enabled organelle slot into. Accepts 'cell', 'nucleus', 'pathogen' and 'cytoplasm'; each writes one &lt;parent&gt;_organelle_summary row per parent with a separate organelle_summary_&lt;slot&gt;_* column family. Raw per-organelle tables are always written when their mask dim is enabled. Default 'cell'; None disables only these rollups.

#### Image Preprocessing (per object)

- **`organelle_rolling_ball`** *(optional)* — (bool) - Apply rolling-ball background estimation with organelle_rolling_ball_radius, subtract the estimated background, and clip negative values to zero before segmentation. This corrects uneven illumination and haze so that a single global threshold can be applied across the field of view, at an additional computational cost per image. Default False.
- **`organelle_rolling_ball_radius`** *(optional)* — (int) - Radius in pixels of the rolling-ball background estimator. It must exceed the diameter of the largest expected organelle to avoid subtracting the objects themselves; an excessively large value may not follow the illumination gradient. An initial value of several times the expected object diameter is appropriate. Default 50; runtime increases steeply with radius.
- **`organelle_clahe`** *(optional)* — (bool) - Rescale each image to 0-1 on its 0.5/99.5 percentiles, then run contrast-limited adaptive histogram equalisation before segmentation. Pulls dim organelles in dark corners up to the same working contrast as bright ones, at the cost of amplifying background noise and destroying absolute intensity comparability between fields. Default False.
- **`organelle_clahe_clip_limit`** *(optional)* — (float) - Contrast ceiling for CLAHE, range 0-1: each tile's histogram is clipped at this height before equalisation, so higher values permit stronger local stretching and more noise amplification. 0.01 is gentle, 0.03-0.05 is aggressive. Only read when organelle_clahe is True. Default 0.01.

#### Object Filtration (all objects)

- **`organelle_min_size`** *(optional)* — (int) - (Deprecated) Minimum object area in square pixels. Most classical segmenters and the U-Net discard smaller components during segmentation via remove_small_objects (the LoG/DoG spot methods do not, and the ring method applies a quarter of it, floor 3, to its edge image), and the value is always applied again to the finished label image. Despite the marker it is still live - raise it to clear dim specks and hot pixels, lower it to keep faint puncta. Default 10; 0 disables.
- **`organelle_max_size`** *(optional)* — (int or None) - Upper area bound in square pixels applied to the final label image; objects above it are deleted rather than split. Use it to remove fused clumps, saturated debris and background regions merged by Otsu into a single component. Values below the largest valid organelle remove biological objects without warning. Default None (no limit).
- **`organelle_min_area`** *(optional)* — (int) - Post-segmentation area floor in square pixels; smaller objects are deleted and the mask relabelled. Raise it to clear noise specks left by thresholding. Default 0 (disabled). Note this is the shared object filter (used by the Qt live preview); the batch organelle mask writer does its own size filtering with organelle_min_size.
- **`organelle_max_area`** *(optional)* — (int or None) - Post-segmentation area ceiling in square pixels; larger objects are deleted. Use it to reject fused clumps and saturated debris. Default 0, and either 0 or None disables it. Note this is the shared object filter (used by the Qt live preview); the batch organelle mask writer caps size with organelle_max_size instead.
- **`organelle_min_object_area`** *(optional)* — (int) - Absolute area threshold in square pixels for the split step. An object is split only when its area exceeds this value in addition to the median-based threshold, preventing division of small organelles into multiple labels. Default 100. Currently inactive because the organelle mask writer does not execute the merge/split stage.
- **`organelle_area_multiplier`** *(optional)* — (float) - Split trigger for organelle_intensity_split: only objects larger than this multiple of the median organelle area in the same field are candidates for watershed splitting. Lower it toward 1.5 to split more aggressively; raise it to restrict splitting to larger organelle aggregates. Default 2.0. Currently inert: the organelle mask writer never runs the merge/split stage.
- **`organelle_min_distance`** *(optional)* — (int) - Minimum separation in pixels between watershed seeds when splitting oversized organelles; distance-transform peaks closer than this collapse into a single seed. Increase it to prevent excessive fragmentation of a single organelle, or decrease it to separate tightly packed puncta. Default 10. Currently inactive because the organelle mask writer does not run the merge/split stage.
- **`organelle_perimeter_fraction`** *(optional)* — (float) - Merge two touching organelle labels when their shared boundary is at least this fraction of the smaller object's perimeter. Range 0-1; increase it toward 1 to merge only nearly fully fused pairs, or decrease it to merge labels with shorter shared boundaries. Default 0 (disabled). Currently inactive because the organelle mask writer does not run the merge/split stage.
- **`organelle_remove_border`** *(optional)* — (bool) - Delete organelle labels touching any image edge during final post-processing so partially imaged objects do not bias area and intensity statistics. This also removes valid objects at the field boundary, with a larger effect for larger organelles. Default False.
- **`organelle_remove_border_objects`** *(optional)* — (bool) - Delete organelle labels touching any image edge during the shared post-segmentation filter (the Qt live preview path). The batch organelle mask writer does the same job from organelle_remove_border, so set that one for a real run. Default False. Enable to keep clipped rim objects out of area and intensity statistics.

#### Intensity Handling (all objects)

- **`organelle_intensity_merge`** *(optional)* — (bool) - Merge two touching organelle labels when the mean intensity along their shared boundary is at least the interior reference of the dimmer object, indicating no dark boundary between them. Use it when thresholding divides one organelle into multiple labels. Default False. This setting is currently inactive because the organelle mask writer does not run the merge/split stage.
- **`organelle_intensity_split`** *(optional)* — (bool) - Split organelle labels whose area exceeds max(organelle_area_multiplier times the median object area, organelle_min_object_area), using a distance-transform watershed seeded by local maxima. Enable when neighbouring puncta are fused into single oversized labels. Default False. Currently inert: the organelle mask writer never runs the merge/split stage.
- **`organelle_intensity_percentile`** *(optional)* — (int) - Percentile (0-100) of an object's interior intensity used as the merge reference when organelle_intensity_threshold_method='percentile'; ignored for 'mean'. Higher values raise the bar the shared boundary must clear, so fewer pairs merge. Default 75. Currently inert: the organelle mask writer never runs the merge/split stage.
- **`organelle_intensity_threshold_method`** *(optional)* — (str) - Reference statistic for organelle_intensity_merge: 'mean' compares the shared-boundary intensity to the mean interior intensity of the dimmer object; 'percentile' compares it to that object's organelle_intensity_percentile value instead. A high percentile makes merging much stricter. Default 'mean'. Currently inert: the organelle mask writer never runs the merge/split stage.
- **`organelle_min_intensity_percentile`** *(optional)* — (int) - Drops organelles whose mean intensity falls below this percentile of all organelle mean intensities in the same field. It is relative, not absolute, so it always removes roughly this share of the dimmest objects even in a clean image. Range 0-100, 0 disables. Default 0. Use it to cull background-level detections.
- **`organelle_max_intensity_percentile`** *(optional)* — (int or None) - Drops organelle objects whose mean intensity exceeds this percentile of all organelle mean intensities in the same field, so it removes roughly the brightest (100 minus value) percent. Range 0-100; 100 or None disables it (None is read as the default 100, it does not error). Applied by the shared Qt live-preview filter - the batch organelle mask pipeline does not run this filter. Default 100. Use it to reject saturated dust and imaging artefacts.

### [`spacr.timelapse.automated_motility_assay`](https://einarolafsson.github.io/spacr/api/spacr/timelapse/index.html#spacr.timelapse.automated_motility_assay)

> This stage inherits `src` from the first stage so that both functions operate on the same dataset.


#### Objects & Channels

- **`src`** *(required)* — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`tracked_object`** *(optional)* — (str) - Which object's feature block ({object}_* columns) the XGBoost infection classifier trains on: 'cell', 'nucleus' or 'pathogen'; anything else falls back to 'cell'. It does not change what is tracked - track geometry and velocity always come from the cell centroids. Default 'cell'.
- **`cell_channel`** *(optional)* — (int or None) - Zero-indexed raw acquisition channel that Cellpose segments into cell masks; it also selects which channel the cell_background, cell_Signal_to_noise and remove_background_cell settings are applied to during preprocessing. Set to None and no cell masks, cell table or cell crops are produced. At least one of cell/nucleus/pathogen/organelle_channel must be an integer or the run aborts. Default None.
- **`nucleus_channel`** *(optional)* — (int or None) - Zero-indexed raw acquisition channel segmented into nucleus masks, and the channel that nucleus_background, nucleus_Signal_to_noise and remove_background_nucleus apply to. None means no nucleus masks, hence no nucleus table, no cell-to-nucleus linking, and nothing subtracted from the cytoplasm mask. Set it whenever a DNA stain was acquired. Default None.
- **`pathogen_channel`** *(optional)* — (int or None) - Zero-indexed raw acquisition channel segmented into pathogen masks (Toxoplasma etc.), and the channel pathogen_background, pathogen_Signal_to_noise and remove_background_pathogen apply to. None disables pathogen segmentation, the pathogen table, the infected-only filter (uninfected) and the adjust_cells step, which needs cell, nucleus and pathogen masks together. Default None.
- **`channels`** *(optional)* — (list of int) - Zero-indexed image channels kept in merged/*.npy and measured by measure_crop; each entry produces its own &lt;object&gt;_channel_&lt;n&gt;_* intensity columns. The list length fixes where masks land, so cell/nucleus/pathogen_mask_dim must shift if you change it. Preprocessing silently resets it to range(n) when it does not match the number of channel folders found. Default [0,1,2,3].

#### Spatial & Temporal Calibration

- **`seconds_per_frame`** *(optional)* — (int) - Interval between consecutive timelapse frames, in seconds. Used with pixels_per_um to convert mean per-frame displacement into um/min; if either is missing, velocities stay in px/frame. It is also printed in the motility plot legend box. A wrong value rescales every reported velocity linearly. Default 60.
- **`pixels_per_um`** *(optional)* — (float) - Image scale in pixels per micrometre. Track coordinates are divided by it, so plots switch from px to um, and together with seconds_per_frame it converts velocity from px/frame to um/min. Take it from the objective and camera pixel size rather than tuning it - it rescales every reported velocity. Default 1.78.

#### Motion Filtering

- **`max_displacement`** *(optional)* — (float) - Largest plausible centroid movement between consecutive frames, in pixels. A single-frame excursion followed by an immediate return is interpolated from neighbouring positions; other displacements above this value cause the complete track to be excluded. Increase the value for rapidly moving objects or sparsely sampled timelapses and decrease it to remove identity-switch artifacts. Default 50.0.
- **`straightness_threshold`** *(optional)* — (float) - Straightness cut-off, where straightness = net displacement / total path length (0 = returns to start, 1 = perfectly straight). When straightness_filter is True, tracks at or above this value are dropped as drift or tracking artifacts, so lowering it discards more tracks. The count is always logged. Default 0.95.
- **`straightness_filter`** *(optional)* — (bool) - Apply the straightness threshold. False reports how many tracks exceed straightness_threshold without changing the data; True removes those tracks from the velocity table, per-well summary and plots. Enable it when stage drift or identity swaps produce implausibly straight trajectories. Default False.
- **`zscore_thresh`** *(optional)* — (float) - Outlier sensitivity when smoothing scalar features within a track (area, bbox area, equivalent diameter, perimeter, solidity, mean/max/min intensity). A frame more than this many standard deviations from its own track mean, whose two neighbours are both within half that, is replaced by their average. Lower smooths more; nothing is deleted. Default 3.0.

#### Infection Classification

- **`infection_intensity_strategy`** *(optional)* — (str) - How infected vs uninfected is decided once infection_intensity_qc is True: 'xgboost' trains a classifier on intensity extremes, 'histogram' picks one intensity threshold, and 'pca'/'umap'/'tsne' cluster a 2D embedding. Unknown values fall back to histogram, as does xgboost when the package is missing or a class is too small. Default 'xgboost'.
- **`infection_intensity_qc_scope`** *(optional)* — (str) - Whether infection QC is fitted once or per group: 'combined'/'global'/'all' fits one model on everything, 'plate'/'per_plate' one per plateID, 'well'/'per_well' one per plate-well, and 'none'/'off' skips QC; an unrecognised string falls back to combined behaviour with a warning. Per-well fitting absorbs staining and exposure differences but needs enough cells per well; every group still writes its own QC plot, only the QC payload embedded in the summary panel is taken from the first processed group. Default 'per_well'.
- **`infection_intensity_mode`** *(optional)* — (str) - Action applied when the quality-control classification disagrees with the mask-based label. 'relabel' replaces the label and retains the cell; 'remove' excludes cells with discordant mask and intensity evidence. Unknown values fall back to 'relabel'. Default 'relabel'.
- **`infection_intensity_n_bins`** *(optional)* — (int) - Bin count for the pathogen-intensity histogram, clamped to 10-256. The histogram strategy evaluates bins from low to high and uses the first bin whose infected fraction reaches the target as the intensity threshold. More bins provide finer threshold resolution but increase variability in per-bin fractions. This setting also controls the QC-panel histogram. Default 64.
- **`db_table_name`** *(optional)* — (str) - Table inside &lt;src&gt;/measurements/measurements.db that holds the pre-QC per-frame measurements. It is rewritten with if_exists='replace' on every run, a companion table with the suffix '_well_motility' holds the well summary, and the same name is read back when reuse_existing_measurements is True. Default 'timelapse_object_measurements'.
- **`reuse_existing_measurements`** *(optional)* — (bool) - If measurements.db already holds the table named by db_table_name, load it instead of re-extracting regionprops from merged/*.npy. Saves most of the runtime when re-running only the infection QC or the plots, but it also skips track smoothing, so changes to max_displacement or zscore_thresh only take effect with this set to False. Default True.
- **`infection_xgb_proba_column`** *(optional)* — (str) - Column used by both the track-level ambiguous filter and the QC probability plot. If it is absent, both components discover the classifier output column, normally infection_prob. Before 2026-08-12 this fallback was unreachable, so the track-level filter was not applied under the default configuration. Set this explicitly only to override discovery. Default 'infection_xgb_proba'.
- **`infection_xgb_drop_ambiguous`** *(optional)* — (bool) - After prediction, discard cells whose probability lies between infection_xgb_ambiguous_low and infection_xgb_ambiguous_high instead of forcing a call on them. True gives cleaner infected vs uninfected motility comparisons at the cost of sample size; False keeps every cell. Only used by the xgboost strategy. Default True.
- **`infection_xgb_ambiguous_low`** *(optional)* — (float) - Lower edge of the discarded probability band, between 0 and 1. Cells whose probability falls between this and infection_xgb_ambiguous_high are dropped when infection_xgb_drop_ambiguous is True. Raise it toward the threshold to keep more cells, lower it to discard more borderline ones. Swapped automatically if it exceeds the high bound. Default 0.25.
- **`infection_xgb_ambiguous_high`** *(optional)* — (float) - Upper edge of the discarded probability band, between 0 and 1. Together with infection_xgb_ambiguous_low it defines the interval whose cells are dropped when infection_xgb_drop_ambiguous is True. Lower it toward the threshold to keep more cells, raise it to discard more. Swapped automatically if it falls below the low bound. Default 0.75.

#### XGBoost Infection Model

- **`infection_xgb_min_cells_per_class`** *(optional)* — (int) - Per well, how many intensity-extreme examples each class must reach before that well's training data are balanced by subsampling to the smaller class; wells that have both classes but fewer examples contribute all of theirs, unbalanced. Wells with only one class are skipped entirely. No well is ever excluded for being small, so raising it leaves more wells unbalanced and the training set more skewed - lower it towards 1 to force balancing in every usable well. Default 10.
- **`infection_xgb_n_estimators`** *(optional)* — (int) - Number of boosting rounds (trees) trained, passed as num_boost_round. More rounds fit the intensity-extreme training set more tightly and push infection probabilities away from 0.5, which shrinks the ambiguous band, but cost runtime and can overfit small wells. Trade off against infection_xgb_learning_rate. Default 200.
- **`infection_xgb_max_depth`** *(optional)* — (int) - Maximum depth of each boosted tree. Deeper trees capture interactions between morphology and pathogen-intensity features but overfit the quartile-derived training labels; shallower trees generalise better across wells. Typical range 2-8; raise it only when the classifier cannot separate infected from uninfected. Default 3.
- **`infection_xgb_learning_rate`** *(optional)* — (float) - Shrinkage applied to each boosting round's contribution (XGBoost eta). Lower values require more rounds but can produce smoother, better-calibrated infection probabilities; higher values converge faster but can yield probabilities concentrated near 0 or 1, reducing the utility of the ambiguous range. Typical range 0.01-0.3; tune together with infection_xgb_n_estimators. Default 0.1.
- **`infection_xgb_subsample`** *(optional)* — (float) - Fraction of training rows drawn at random for each boosting round, between 0 and 1. Below 1 it injects stochasticity that limits overfitting to the small set of intensity-extreme cells used for training; 1.0 uses every training row every round. Lower it if the classifier appears to memorise individual wells. Default 0.8.
- **`infection_xgb_colsample_bytree`** *(optional)* — (float) - Fraction of feature columns offered to each tree, between 0 and 1. Lowering it stops a couple of dominant pathogen-intensity features from being chosen by every tree, spreading gain across morphology features and reducing overfitting; 1.0 exposes all features to every tree. Default 0.8.
- **`infection_xgb_reg_lambda`** *(optional)* — (float) - L2 penalty on leaf weights. Larger values shrink leaf outputs, giving a more conservative model whose probabilities sit closer to 0.5 and therefore more cells inside the ambiguous band; 0 removes the penalty entirely. Raise it when the model fits training cells perfectly yet disagrees wildly with mask-based labels. Default 1.0.
- **`infection_xgb_random_state`** *(optional)* — (int) - Seed for the generator that balances the per-well training set, i.e. which intensity-extreme cells are sampled for each class. It is not handed to XGBoost itself. Change it and re-run to confirm the adjusted infection calls are stable under a different training draw. Default 42.
- **`infection_xgb_n_jobs`** *(optional)* — (int) - Threads XGBoost uses for training and prediction (its nthread parameter). -1 uses every available core; set a small positive number to leave CPU free for other work or when several plates run at once. It changes runtime, not the training recipe. Default -1.
- **`infection_xgb_proba_threshold`** *(optional)* — (float) - Predicted probability at or above which a cell is called infected, between 0 and 1. Lowering it makes infection calling more permissive (more cells become infected), raising it more stringent. It is also the centre of the confidence band whose half-width is infection_xgb_margin. Default 0.5.
- **`infection_xgb_margin`** *(optional)* — (float) - Half-width of the confidence band around infection_xgb_proba_threshold, clamped to 0-0.49. In 'relabel' mode only cells outside the band get their label overridden, the rest keep the mask-based call; in 'remove' mode cells inside the band are spared deletion. Raise it to trust the model less. Default 0.15.
- **`infection_xgb_top_features`** *(optional)* — (int) - How many features, ranked by XGBoost gain, are retained for the feature-importance panel of the QC figure. This is a display cut applied after training: it never changes the model or the infection calls. Lower it for a readable bar chart, raise it to inspect more features. Default 20.

#### Infection Clustering

- **`infection_pca_n_clusters`** *(optional)* — (int) - Intended cluster count for the embedding-based infection call. Not currently honoured: the pca/umap/tsne QC always runs KMeans with exactly two clusters, one mapped to infected and one to uninfected, so changing this has no effect on results. Default 2.
- **`infection_pca_random_state`** *(optional)* — (int) - Seed for KMeans and for the UMAP/t-SNE embeddings in the pca/umap/tsne strategies. Fixing it makes the embedding and the resulting infected/uninfected cluster assignment reproducible; change it to check that the split is not an artifact of one initialisation. Note the max-cells subsample uses its own fixed seed. Default 42.
- **`infection_pca_pathogen_weight`** *(optional)* — (float) - Multiplier applied to the standardised pathogen-channel features before embedding. Above 1 it stretches the embedding along pathogen intensity so KMeans splits infected from uninfected rather than by morphology; 1.0 leaves all features weighted equally. Raise it when the log reports weak cluster separation. Default 2.0.
- **`infection_pca_log_intensity`** *(optional)* — (bool) - Apply log1p to non-negative features whose names contain 'intensity', 'p75', 'p95', or 'max' before standardization and embedding. This compresses the upper tail and reduces the influence of a small number of high-intensity cells. Consider enabling for pathogen stains with a wide dynamic range. Default False.
- **`infection_pca_min_silhouette`** *(optional)* — (float) - Silhouette value below which the log prints a 'weak cluster structure' warning with tuning hints. It does not reject or re-run the clustering - the cluster-derived labels are applied regardless - so treat it purely as an alert level. Silhouette runs from -1 to 1. Default 0.05.
- **`infection_pca_min_gt_separation`** *(optional)* — (float) - Alert level for the ground-truth separation score - the absolute difference, between the two clusters, in the fraction of intensity-extreme cells that are infected (0-1). Dropping below it only prints a warning; the cluster labels are still applied. Raise it to be told sooner that the embedding is not separating infection. Default 0.2.
- **`infection_pca_max_cells`** *(optional)* — (int) - Maximum number of cells included in the embedding. When more cells are available, a random subsample of this size is drawn with a fixed seed of 0, independently of infection_pca_random_state. Decrease the value to reduce UMAP or t-SNE runtime and memory use; increase it to improve representation of rare subpopulations. Applied after removal of non-finite rows. Default 50000.

#### Embedding Search

- **`infection_pca_umap_search`** *(optional)* — (bool) - Fit UMAP once per combination of infection_pca_umap_n_neighbors_grid and infection_pca_umap_min_dist_grid, keeping the run with the highest cluster-centroid distance times ground-truth separation. True costs one UMAP fit per grid point; False does a single fit using infection_pca_umap_n_neighbors and infection_pca_umap_min_dist. Default True.
- **`infection_pca_umap_n_neighbors_grid`** *(optional)* — (list[int]) - Candidate UMAP n_neighbors values tried when infection_pca_umap_search is True. Small values (around 5) preserve local structure and split fine subpopulations; large values (30 and up) emphasise global structure. Every entry is paired with every value in infection_pca_umap_min_dist_grid, so keep the list short. Default [5, 10, 15, 30].
- **`infection_pca_umap_min_dist_grid`** *(optional)* — (list[float]) - Candidate UMAP min_dist values tried when infection_pca_umap_search is True, each between 0 and 1. Near 0 packs points tightly and gives crisper clusters for KMeans to split; larger values spread points out and blur the boundary. Paired with every n_neighbors candidate. Default [0.0, 0.05, 0.1, 0.3].
- **`infection_pca_umap_n_neighbors`** *(optional)* — (int) - Fixed UMAP n_neighbors used when infection_pca_umap_search is False; it defines the size of the local neighbourhood UMAP attempts to preserve. Low values (5-10) emphasize local detail and can divide one population into multiple clusters; values of 30 or greater emphasize global structure. Ignored during grid search. Default 15.
- **`infection_pca_umap_min_dist`** *(optional)* — (float) - Fixed UMAP min_dist used when infection_pca_umap_search is False, between 0 and 1: the minimum spacing allowed between embedded points. Near 0 gives tight, well-separated clumps that KMeans splits cleanly; larger values spread points evenly and blur the infected/uninfected boundary. Ignored during grid search. Default 0.1.
- **`infection_pca_tsne_search`** *(optional)* — (bool) - Fit t-SNE once per combination of infection_pca_tsne_perplexity_grid and infection_pca_tsne_learning_rate_grid, keeping the run that scores highest on centroid distance times ground-truth separation. False does a single fit at infection_pca_tsne_perplexity with learning_rate 'auto'. Every extra grid point costs a full t-SNE fit. Default True.
- **`infection_pca_tsne_perplexity_grid`** *(optional)* — (list[float]) - Candidate t-SNE perplexity values tried when infection_pca_tsne_search is True - roughly how many neighbours each point balances. Candidates at or above (n_cells-1)/3 are discarded, and if none survive the code falls back to min(30, that cap). Small values fragment clusters, large ones merge them. Default [15.0, 30.0, 45.0].
- **`infection_pca_tsne_learning_rate_grid`** *(optional)* — (list[float]) - Candidate t-SNE learning rates tried when infection_pca_tsne_search is True. Too low leaves a dense ball with points crowded together; too high scatters the map into a diffuse cloud. Either way the infected/uninfected split blurs. Paired with every perplexity candidate, so keep both lists short. Default [200.0, 500.0].
- **`infection_pca_tsne_perplexity`** *(optional)* — (float) - Fixed t-SNE perplexity used when infection_pca_tsne_search is False, automatically capped at max(5, (n_cells-1)/3). Lower values emphasise local structure and can break one population into several clumps; higher values emphasise global structure and merge them. The learning rate is left at 'auto'. Default 30.0.

#### Motility Plots & QC

- **`motility_ylim`** *(optional)* — (tuple) - Spatial y-axis limits for the origin-centred track panels (infected and uninfected) of the motility figure, in plotted coordinate units - um when pixels_per_um is set, otherwise pixels - not velocity. The whole-field all-tracks axis next to them always autoscales from the data and ignores this setting. Set to None for autoscaling. Default (100, -100), a 200-unit window written high-to-low so the axis draws reversed.
- **`motility_xlim`** *(optional)* — (tuple) - Spatial x-axis limits for the origin-centred track panels (infected and uninfected) of the motility figure, in plotted coordinate units - um when pixels_per_um is set, otherwise pixels - not time. The whole-field all-tracks axis next to them always autoscales from the data and ignores this setting. Set to None for autoscaling. Default (100, -100), a 200-unit window written high-to-low so the axis draws reversed.
- **`infection_intensity_qc_graphs`** *(optional)* — (bool) - Save the infection-intensity histogram PNG and reserve the QC sub-axes (histogram, embedding, or XGBoost probability plus feature importance) inside the combined intensity/motility panel. Set False to skip that plotting work on large runs; the infection relabelling itself is unchanged either way. Default True.

#### Runtime & Reliability

- **`n_jobs`** *(optional)* — (int) - CPU workers for parallel stages: measurement, mask adjustment, DataLoader loading, and the sklearn/UMAP calls where -1 means every core. Raise it to shorten CPU-bound steps until RAM or disk I/O saturates. Note the measure-and-crop pipeline overrides your value with cpu_count()-4. Defaults vary by pipeline: cpu_count()-4, -1, or None.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
preprocess_generate_masks_timelapse_settings = {
    # Input & Metadata
    # Required settings
    'src': 'path',
    # Conditionally required settings
    'cell_channel': None,
    'nucleus_channel': None,
    'pathogen_channel': None,
    # Optional settings
    'channels': [0, 1, 2, 3],
    'magnification': 20,
    'metadata_type': 'cellvoyager',
    'custom_regex': None,

    # Acquisition & Axes
    # Optional settings
    'z_stack': False,
    'z_segmentation_mode': 'project',
    'z_axis': None,
    'z_projection': 'max',
    'anisotropy': None,
    'voxel_size_z_um': None,
    'voxel_size_xy_um': None,
    'stitch_threshold': 0.25,

    # Image Preprocessing
    # Optional settings
    'normalize': True,
    'lower_percentile': 2,
    'randomize': True,
    'batch_fields': 8,
    'consolidate': False,
    'denoise': False,

    # Cell Segmentation
    # Optional settings
    'cell_model_name': 'cpsam',
    'cell_diameter': None,
    'cell_CP_prob': 0,
    'cell_FT': 1.0,
    'adjust_cells': False,

    # Nucleus Segmentation
    # Optional settings
    'nucleus_model_name': 'cpsam',
    'nucleus_diameter': None,
    'nucleus_CP_prob': 0,
    'nucleus_FT': 1.0,

    # Pathogen Segmentation
    # Optional settings
    'pathogen_model_name': 'cpsam',
    'pathogen_diameter': None,
    'pathogen_CP_prob': 0,
    'pathogen_FT': 1.0,
    'pathogen_model': None,

    # Image Preprocessing (per object)
    # Optional settings
    'cell_background': 100,
    'cell_Signal_to_noise': 10,
    'remove_background_cell': False,
    'nucleus_background': 100,
    'nucleus_Signal_to_noise': 10,
    'remove_background_nucleus': False,
    'pathogen_background': 100,
    'pathogen_Signal_to_noise': 10,
    'remove_background_pathogen': False,

    # Object Filtration (all objects)
    # Optional settings
    'cell_min_area': 0,
    'cell_max_area': 0,
    'cell_min_object_area': 100,
    'cell_area_multiplier': 2.0,
    'cell_min_distance': 10,
    'cell_perimeter_fraction': 0,
    'cell_remove_border_objects': False,
    'nucleus_min_area': 0,
    'nucleus_max_area': 0,
    'nucleus_min_object_area': 100,
    'nucleus_area_multiplier': 2.0,
    'nucleus_min_distance': 10,
    'nucleus_perimeter_fraction': 0,
    'nucleus_remove_border_objects': False,
    'pathogen_min_area': 0,
    'pathogen_max_area': 0,
    'pathogen_min_object_area': 100,
    'pathogen_area_multiplier': 2.0,
    'pathogen_min_distance': 10,
    'pathogen_perimeter_fraction': 0,
    'pathogen_remove_border_objects': False,

    # Intensity Handling (all objects)
    # Optional settings
    'cell_intensity_merge': False,
    'cell_intensity_split': False,
    'cell_intensity_percentile': 75,
    'cell_intensity_threshold_method': 'mean',
    'cell_min_intensity_percentile': 0,
    'cell_max_intensity_percentile': 100,
    'nucleus_intensity_merge': False,
    'nucleus_intensity_split': False,
    'nucleus_intensity_percentile': 75,
    'nucleus_intensity_threshold_method': 'mean',
    'nucleus_min_intensity_percentile': 0,
    'nucleus_max_intensity_percentile': 100,
    'pathogen_intensity_merge': False,
    'pathogen_intensity_split': False,
    'pathogen_intensity_percentile': 75,
    'pathogen_intensity_threshold_method': 'mean',
    'pathogen_min_intensity_percentile': 0,
    'pathogen_max_intensity_percentile': 100,

    # Quality Control
    # Optional settings
    'seg_qc': 'report',
    'seg_qc_min_objects': 10,
    'seg_qc_count_ratio': 0.25,
    'seg_qc_size_ratio': 1.4,
    'seg_qc_border_fraction': 0.3,
    'seg_qc_outlier_mad': 5.0,
    'seg_qc_outlier_fraction': 0.15,
    'seg_qc_foreground_fraction': 0.35,
    'seg_qc_split_ratio': 2.0,
    'seg_qc_min_diameter': 5.0,
    'seg_qc_tiny_fraction': 0.3,
    'seg_qc_max_object_fraction': 0.25,
    'seg_qc_plate_fail_fraction': 0.1,

    # Visualization & Diagnostics
    # Optional settings
    'plot': False,
    'cmap': 'inferno',
    'figuresize': 10,
    'normalize_plots': True,
    'examples_to_plot': 1,

    # Output & Storage
    # Optional settings
    'save': True,
    'delete_intermediate': False,
    'keep_intermediate': False,
    'keep_original_images': False,
    'save_original_images': True,
    'keep_npz': False,
    'filter': False,
    'merge_pathogens': False,

    # Runtime & Reliability
    # Optional settings
    'preprocess': True,
    'masks': True,
    'test_mode': False,
    'test_images': 10,
    'resume': False,
    'strict_errors': None,
    'max_failure_rate': None,
    'dry_run': False,
    'verbose': True,
    'n_jobs': max(1, (__import__('os').cpu_count() or 1) - 4),
    'batch_size': 50,
    'pipeline_style': 'v1',
    'diameter_estimate_n_fields': 5,
}

automated_motility_assay_settings = {
    # Objects & Channels
    # Required settings
    'src': preprocess_generate_masks_timelapse_settings['src'],
    # Optional settings
    'tracked_object': 'cell',
    'cell_channel': 2,
    'nucleus_channel': 0,
    'pathogen_channel': 1,
    'channels': [0, 1, 2, 3],

    # Spatial & Temporal Calibration
    # Optional settings
    'seconds_per_frame': 60,
    'pixels_per_um': 1.78,

    # Motion Filtering
    # Optional settings
    'max_displacement': 50.0,
    'straightness_threshold': 0.95,
    'straightness_filter': False,
    'zscore_thresh': 3.0,

    # Infection Classification
    # Optional settings
    'infection_intensity_strategy': 'xgboost',
    'infection_intensity_qc_scope': 'per_well',
    'infection_intensity_mode': 'relabel',
    'infection_intensity_n_bins': 64,
    'db_table_name': 'timelapse_object_measurements',
    'reuse_existing_measurements': True,
    'infection_xgb_proba_column': 'infection_xgb_proba',
    'infection_xgb_drop_ambiguous': True,
    'infection_xgb_ambiguous_low': 0.25,
    'infection_xgb_ambiguous_high': 0.75,

    # XGBoost Infection Model
    # Optional settings
    'infection_xgb_min_cells_per_class': 10,
    'infection_xgb_n_estimators': 200,
    'infection_xgb_max_depth': 3,
    'infection_xgb_learning_rate': 0.1,
    'infection_xgb_subsample': 0.8,
    'infection_xgb_colsample_bytree': 0.8,
    'infection_xgb_reg_lambda': 1.0,
    'infection_xgb_random_state': 42,
    'infection_xgb_n_jobs': -1,
    'infection_xgb_proba_threshold': 0.5,
    'infection_xgb_margin': 0.15,
    'infection_xgb_top_features': 20,

    # Infection Clustering
    # Optional settings
    'infection_pca_n_clusters': 2,
    'infection_pca_random_state': 42,
    'infection_pca_pathogen_weight': 2.0,
    'infection_pca_log_intensity': False,
    'infection_pca_min_silhouette': 0.05,
    'infection_pca_min_gt_separation': 0.2,
    'infection_pca_max_cells': 50000,

    # Embedding Search
    # Optional settings
    'infection_pca_umap_search': True,
    'infection_pca_umap_n_neighbors_grid': [5, 10, 15, 30],
    'infection_pca_umap_min_dist_grid': [0.0, 0.05, 0.1, 0.3],
    'infection_pca_umap_n_neighbors': 15,
    'infection_pca_umap_min_dist': 0.1,
    'infection_pca_tsne_search': True,
    'infection_pca_tsne_perplexity_grid': [15.0, 30.0, 45.0],
    'infection_pca_tsne_learning_rate_grid': [200.0, 500.0],
    'infection_pca_tsne_perplexity': 30.0,

    # Motility Plots & QC
    # Optional settings
    'motility_ylim': (100, -100),
    'motility_xlim': (100, -100),
    'infection_intensity_qc_graphs': True,

    # Runtime & Reliability
    # Optional settings
    'n_jobs': 8,
}

In [ ]:
preprocess_generate_masks_timelapse_settings.update({
    # Input & Metadata
    # Conditionally required settings
    'organelle_channel': None,
    # Optional settings
    'number_of_organelles': 0,

    # Organelle Segmentation
    # Optional settings
    'organelle_type': 'custom',
    'organelle_diameter': 30,
    'organelle_mask_within_cells': False,

    # Organelle Segmentation (advanced)
    # Optional settings
    'organelle_morphology': 'spots',
    'organelle_method': 'otsu',
    'organelle_adaptive_block_size': 51,
    'organelle_adaptive_offset': 5,
    'organelle_tophat_radius': 5,
    'organelle_watershed_spots': True,
    'organelle_log_min_sigma': 1,
    'organelle_log_max_sigma': 10,
    'organelle_log_num_sigma': 10,
    'organelle_log_threshold': 0.01,
    'organelle_dog_sigma_low': 1.0,
    'organelle_dog_sigma_high': 3.0,
    'organelle_ridge_filter': 'frangi',
    'organelle_ridge_sigmas': [1, 2, 3],
    'organelle_skeletonize': False,
    'organelle_network_threshold': 'otsu',
    'organelle_hysteresis_low': 0.2,
    'organelle_hysteresis_high': 0.6,
    'organelle_ring_sigma_inner': 1.0,
    'organelle_ring_sigma_outer': 3.0,
    'organelle_ring_min_prominence': 0.1,
    'organelle_ring_fill_method': 'flood',
    'organelle_morph_radius': 3,
    'organelle_fill_holes': 64,
    'organelle_model_name': 'cpsam',
    'organelle_CP_prob': 0.0,
    'organelle_FT': 0.4,
    'organelle_resample': True,
    'organelle_unet_model_path': None,
    'organelle_unet_threshold': 0.5,
    'summarize_organelles_by': 'cell',

    # Image Preprocessing (per object)
    # Optional settings
    'organelle_rolling_ball': False,
    'organelle_rolling_ball_radius': 50,
    'organelle_clahe': False,
    'organelle_clahe_clip_limit': 0.01,

    # Object Filtration (all objects)
    # Optional settings
    'organelle_min_size': 10,
    'organelle_max_size': None,
    'organelle_min_area': 0,
    'organelle_max_area': 0,
    'organelle_min_object_area': 100,
    'organelle_area_multiplier': 2.0,
    'organelle_min_distance': 10,
    'organelle_perimeter_fraction': 0,
    'organelle_remove_border': False,
    'organelle_remove_border_objects': False,

    # Intensity Handling (all objects)
    # Optional settings
    'organelle_intensity_merge': False,
    'organelle_intensity_split': False,
    'organelle_intensity_percentile': 75,
    'organelle_intensity_threshold_method': 'mean',
    'organelle_min_intensity_percentile': 0,
    'organelle_max_intensity_percentile': 100,
})


In [ ]:
preprocess_generate_masks_timelapse(preprocess_generate_masks_timelapse_settings)
automated_motility_assay(automated_motility_assay_settings)

## Outputs and next steps

Tracked masks, per-track measurements, well-level motility summaries, and quality-control figures.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)